# Practical 0 — The Python You Need Before the CSE276 Practicals

### CSE276 · Artificial Intelligence Foundations

Practical 1 builds a **route planner** with BFS and DFS.
Practical 2 solves the **8-puzzle** with Hill Climbing, Best First Search and A\*.

Neither practical teaches Python — they assume it. **This notebook is that assumption, written down.**
Every feature below appears in Practical 1 or Practical 2. Nothing extra is included.

---

### How to use this notebook

1. Run every cell from top to bottom with **Shift + Enter**.
2. Read the output and check it is what you expected.
3. Where a cell says **"Your turn"**, change the code and run it again.
4. Finish with the **self-check** in section 17 and the **checklist** in section 19.

If you can do everything in the checklist, you are ready for the practicals.

---

### What each practical needs

| Topic | Section | Practical 1 | Practical 2 |
|---|---|---|---|
| Variables, `print()`, f-strings | 1 | ✔ | ✔ |
| Lists — index, slice, append, `+` | 2 | ✔ | ✔ |
| Tuples, and why a board is one | 3 | | ✔ |
| Dictionaries — storing a graph | 4 | ✔ | ✔ |
| Sets — the `visited` trick | 5 | ✔ | ✔ |
| `if` / `in` / `not in` | 6 | ✔ | ✔ |
| `for` and `while` loops | 7, 8 | ✔ | ✔ |
| Functions, returning two values | 9 | ✔ | ✔ |
| `deque` — queue and stack | 10 | ✔ | |
| `heapq` — priority queue | 11 | | ✔ |
| `min(..., key=...)` and `lambda` | 12 | | ✔ |
| `//` and `%` for grid positions | 13 | | ✔ |
| Swapping values | 14 | | ✔ |
| Printing a grid | 15 | | ✔ |

## 1. Variables, `print()` and f-strings

A variable is a name for a value. Python works out the type for you.

In [ ]:
name = "Main Gate"       # a string  (str)
distance = 300           # a whole number (int)
cost = 4.5               # a decimal number (float)
found = True             # True or False (bool)
nothing = None           # "no value at all"

print(name, distance, cost, found, nothing)
print()

# type() tells you what something is
print("type of name     :", type(name))
print("type of distance :", type(distance))

# An f-string lets you drop a value straight into text.
# Put f before the quotes, then {} around the value.
print()
print(f"Walking to {name} costs {distance} metres.")
print(f"Two steps would be {distance * 2} metres.")

> **Where you meet this:** every `print()` in both practicals, e.g.
> `print(f"Exploring: {current}")` in Practical 1.

## 2. Lists — an ordered, changeable sequence

A **list** holds items in order. In the practicals a list is used for a **path**:
`["Main Gate", "Hostel", "Library"]`.

```text
   path = [ "Main Gate" , "Hostel" , "Library" ]
index:          0            1           2
                                        -1   <- last item
```

In [ ]:
path = ["Main Gate", "Hostel", "Library"]

print("The whole list :", path)
print("How many items :", len(path))
print("First item     :", path[0])
print("Last item      :", path[-1])     # -1 means "the last one" - used constantly
print("Middle onwards :", path[1:])     # a slice: from index 1 to the end
print()

# append() adds one item to the END of the same list
path.append("Canteen")
print("After append   :", path)

# "+" builds a NEW list and leaves the original alone.
# This is how a search grows a path without damaging the old one.
old_path = ["Main Gate", "Hostel"]
new_path = old_path + ["Library"]

print()
print("old_path stays :", old_path)
print("new_path is    :", new_path)

> **Why `path + [neighbour]` and not `path.append(neighbour)`?**
> A search holds many paths at once. `append` would change a path that other
> branches are still using. `+` makes a fresh copy, so each branch keeps its own history.
> This exact line appears in Practical 1:
> `new_path = path + [neighbour]`

## 3. Tuples — like a list, but frozen

A **tuple** uses round brackets and **cannot be changed** after it is made.

Practical 2 stores an 8-puzzle board as a tuple of 9 numbers:

```text
   1  3  6
   5  _  2     ->   (1, 3, 6, 5, 0, 2, 4, 7, 8)
   4  7  8
```

**Why a tuple and not a list?** Because only unchangeable things can be put in a
`set` or used as a dictionary key — and search algorithms need exactly that to
remember which boards they have already seen.

In [ ]:
board = (1, 3, 6,
         5, 0, 2,
         4, 7, 8)

print("Board        :", board)
print("Item 0       :", board[0])
print("Items 3 to 5 :", board[3:6])     # slicing works the same as a list
print("Where is 0   :", board.index(0)) # .index() finds the position of a value
print("How many     :", len(board))

# A tuple cannot be changed - this is the whole point
try:
    board[0] = 9
except TypeError as error:
    print()
    print("Trying to change a tuple gives:", error)

# A tuple CAN go in a set. A list cannot.
seen = set()
seen.add(board)
print()
print("A tuple can be stored in a set :", board in seen)

try:
    seen.add([1, 2, 3])
except TypeError as error:
    print("A list cannot:", error)

## 4. Dictionaries — how the campus map is stored

A **dictionary** maps a **key** to a **value**. Practical 1 stores the campus as a
dictionary where each key is a location and each value is a list of its neighbours.
This is called an **adjacency list**.

```text
   "Main Gate"  ->  ["Hostel", "Admin", "Lab"]
        key                   value
```

In [ ]:
campus = {
    "Main Gate": ["Hostel", "Admin", "Lab"],
    "Hostel":    ["Main Gate", "Library", "Canteen"],
    "Library":   ["Hostel", "Canteen"],
    "Canteen":   ["Hostel", "Library", "Admin"],
    "Admin":     ["Main Gate", "Canteen", "Lab"],
    "Lab":       ["Main Gate", "Admin"]
}

# Look a key up with square brackets
print("Neighbours of Main Gate :", campus["Main Gate"])
print("Neighbours of Library   :", campus["Library"])
print()

# .items() gives you the key AND the value together
print("The whole map:")
for location, neighbours in campus.items():
    print(f"  {location} -> {neighbours}")

print()
# "in" checks whether a KEY exists
print("Is 'Canteen' a place?", "Canteen" in campus)
print("Is 'Gym' a place?    ", "Gym" in campus)

# A dictionary is also how the practicals collect results at the end
results = {"BFS": 3, "DFS": 5}
results["A*"] = 3                    # add a new key
print()
print("Results:", results)

## 5. Sets — remembering what you have already seen

A **set** holds items with **no duplicates and no order**. Every search in both
practicals keeps one, called `visited`, so it never explores the same place twice.

The important property: checking `x in my_set` is **fast no matter how big the set is**,
while `x in my_list` has to look through the list item by item.

In [ ]:
visited = set()          # start empty - note set(), not {}

visited.add("Main Gate")
visited.add("Hostel")
visited.add("Main Gate")  # adding it twice changes nothing

print("visited     :", visited)
print("How many    :", len(visited))
print()

# This is the exact test every search does
current = "Hostel"
if current in visited:
    print(current, "-> already seen, skip it")
else:
    print(current, "-> new, explore it")

print()
current = "Library"
print("Is Library visited?", current in visited)
print("Is Library NEW?    ", current not in visited)

> **Where you meet this** (Practical 1, inside `bfs`):
> ```python
> if current in visited:
>     continue
> visited.add(current)
> ```

## 6. Making decisions — `if`, `elif`, `else`

A **goal test** is just an `if`. Note the double `==` for *"is equal to"*;
a single `=` means *"assign"*.

In [ ]:
current = "Library"
goal = "Library"

if current == goal:
    print("Goal reached!")
else:
    print("Not there yet, keep searching.")

print()

h = 4                       # an imaginary heuristic value
if h == 0:
    print("h = 0  -> we are at the goal")
elif h < 3:
    print("h < 3  -> very close")
else:
    print("h is still large -> a long way to go")

print()
# and / or / not
explored = 12
if h > 0 and explored < 100:
    print("Not finished, and we still have budget to keep searching.")

# Indentation is not decoration - it is what tells Python
# which lines belong inside the if.

## 7. `for` loops — doing something to every item

In [ ]:
# 1. Loop over a list
for neighbour in ["Hostel", "Admin", "Lab"]:
    print("Neighbour:", neighbour)

print()
# 2. Loop over numbers. range(9) gives 0,1,2,...,8  (9 is NOT included)
for i in range(9):
    print(i, end=" ")
print()
print()

# 3. Loop with the position as well, using enumerate()
path = ["Main Gate", "Hostel", "Library"]
for step, place in enumerate(path):
    print(f"Step {step}: {place}")

print()
# 4. continue = skip the rest of THIS round;  break = leave the loop early
for i in range(6):
    if i == 2:
        continue          # skip printing 2
    if i == 5:
        break             # stop completely
    print(i, end=" ")
print()

## 8. `while` loops — and the trick that runs every search

Every search in both practicals is built on this one line:

```python
while queue:
```

An **empty list, set, deque or dictionary is treated as `False`**; a non-empty one is `True`.
So `while queue:` means *"keep going while there is still something left to explore"*.

In [ ]:
# Proof of the trick
print("Is an empty list True? ", bool([]))
print("Is [1, 2] True?        ", bool([1, 2]))
print("Is an empty set True?  ", bool(set()))
print()

# The shape of EVERY search loop you are about to write
frontier = ["A", "B", "C"]

while frontier:                     # keep going while the frontier is not empty
    item = frontier.pop(0)          # take one out
    print("Processing", item, " | left:", frontier)

print("Frontier is empty - the loop ends by itself.")

## 9. Functions — `def`, parameters and `return`

A function is a named block of code you can run again with different inputs.
Both practicals define `bfs(graph, start, goal)`, `h_manhattan(state)` and so on.

In [ ]:
def steps_needed(path):
    """Number of moves in a path (a 3-place path is 2 moves)."""
    return len(path) - 1


print("Moves in a 3-place path:", steps_needed(["A", "B", "C"]))
print()


# A function can return TWO values - the practicals use this a lot
def search(goal_found):
    path = ["Main Gate", "Library"]
    explored = ["Main Gate", "Hostel", "Library"]
    if goal_found:
        return path, explored        # returns a pair
    return None, explored            # None means "no path found"


# Unpack the pair into two names
found_path, explored_list = search(True)
print("path     :", found_path)
print("explored :", explored_list)

print()
missing_path, explored_list = search(False)
print("path     :", missing_path)

# Always test for None BEFORE using a result that might be missing
if missing_path is None:
    print("No path was found, so we must not call len() on it.")
else:
    print("Moves:", steps_needed(missing_path))

> **A function that ends without `return` gives back `None`.** That is why
> Practical 1 finishes `bfs` with `return None, explored` — it is saying *"no path exists"*.

## 10. `deque` — the Queue (BFS) and the Stack (DFS)

This is the **only** difference between BFS and DFS in Practical 1.

```text
QUEUE  (FIFO - first in, first out)      STACK  (LIFO - last in, first out)

 in ->  [A][B][C]  -> out                       out <-  [A][B][C]  <- in
        popleft() takes A                               pop() takes C
```

In [ ]:
from collections import deque

# ---------- QUEUE: what BFS uses ----------
queue = deque()
queue.append("A")
queue.append("B")
queue.append("C")
print("Queue        :", list(queue))
print("popleft()    :", queue.popleft(), " <- the OLDEST item leaves first")
print("Queue now    :", list(queue))

print()
# ---------- STACK: what DFS uses ----------
stack = []
stack.append("A")
stack.append("B")
stack.append("C")
print("Stack        :", stack)
print("pop()        :", stack.pop(), " <- the NEWEST item leaves first")
print("Stack now    :", stack)

print()
print("BFS = deque + popleft() = FIFO")
print("DFS = list  + pop()     = LIFO")

## 11. `heapq` — the Priority Queue (A\*)

A queue and a stack both ignore **how good** an item is. Informed search needs the
**best** item out next, whatever order things went in. That is a **priority queue**.

`heapq` turns a normal list into one:

- `heappush(list, item)` adds an item
- `heappop(list)` removes the item with the **smallest** first value

Items are usually tuples like `(score, thing)` — Python compares the first element first.

In [ ]:
import heapq

pq = []
heapq.heappush(pq, (9, "Board A"))
heapq.heappush(pq, (4, "Board B"))
heapq.heappush(pq, (7, "Board C"))

print("Added: A with 9, B with 4, C with 7")
print()
print("Taking items out - smallest score always leaves first:")
while pq:
    score, name = heapq.heappop(pq)      # unpack the tuple into two names
    print(f"   {name} (score {score})")

> **The whole pattern in one line each:**
> `BFS = Queue = FIFO` · `DFS = Stack = LIFO` · `A* = Priority Queue = best first`

## 12. `min()` with `key=` and `lambda`

`min()` normally returns the smallest value. Add `key=` and it returns the item whose
**score** is smallest — the score being whatever function you give it.

A `lambda` is just a one-line function written inline.

In [ ]:
numbers = [7, 2, 9]
print("Plain min():", min(numbers))
print()

# Score each board with a made-up heuristic
def h(board):
    return sum(board)          # pretend "sum of tiles" is our estimate

boards = [(1, 2, 9), (1, 2, 3), (4, 5, 6)]

best = min(boards, key=h)      # the board with the smallest h
print("Boards      :", boards)
print("Best board  :", best, "with h =", h(best))

print()
# The same thing written with a lambda (no separate def needed)
best = min(boards, key=lambda b: sum(b))
print("With lambda :", best)

print()
# This is exactly how the simplified A* picks a path:
#     path = min(frontier, key=lambda p: (len(p) - 1) + h(p[-1]))
frontier = [["A"], ["A", "B"], ["A", "B", "C"]]
shortest = min(frontier, key=lambda p: len(p))
print("Shortest path in the frontier:", shortest)

## 13. `//` and `%` — turning a position into a row and column

The 8-puzzle board is **nine numbers in a row**, but we think of it as a **3 × 3 grid**.
These two operators convert between the two views, and Practical 2 uses them constantly.

```text
position:   0  1  2        row = position // 3      (whole-number division)
            3  4  5        col = position %  3      (remainder)
            6  7  8

position 5  ->  row = 5 // 3 = 1 ,  col = 5 % 3 = 2
row 1, col 2 -> position = 1 * 3 + 2 = 5
```

In [ ]:
print(" 7 / 3  =", 7 / 3,  "  <- normal division gives a decimal")
print(" 7 // 3 =", 7 // 3, "  <- floor division throws the decimal away")
print(" 7 %  3 =", 7 % 3,  "  <- modulo gives the remainder")
print()

print("pos | row | col")
for position in range(9):
    row = position // 3
    col = position % 3
    print(f" {position}  |  {row}  |  {col}")

print()
# ...and back again
row, col = 2, 1
print(f"row {row}, col {col} -> position {row * 3 + col}")

print()
# Staying on the board: a move is legal only if the new row and column are 0, 1 or 2
new_row, new_col = 3, 1
if 0 <= new_row <= 2 and 0 <= new_col <= 2:
    print("Move is on the board")
else:
    print(f"row {new_row} is off the board - reject this move")

## 14. Swapping two values — how a tile slides

A move in the 8-puzzle is one swap: the blank and the tile next to it change places.
Python swaps in a single line, with no temporary variable.

In [ ]:
a, b = 10, 20
print("before:", a, b)
a, b = b, a                 # the swap
print("after :", a, b)

print()
# Making a new board from an old one:
#   1. tuple -> list, because tuples cannot be changed
#   2. swap
#   3. list -> tuple, so the result can go in a set
board = (1, 3, 6,
         5, 0, 2,
         4, 7, 8)

blank = board.index(0)      # where the blank is
target = blank + 1          # slide the tile on its right into it

new_board = list(board)                                  # 1
new_board[blank], new_board[target] = new_board[target], new_board[blank]   # 2
new_board = tuple(new_board)                             # 3

print("old board:", board)
print("new board:", new_board)
print("old board is untouched:", board)

## 15. Printing a board so a human can read it

Two tools: **slicing** to take three numbers at a time, and **`" ".join()`** to put them
on one line with spaces between.

In [ ]:
board = (1, 3, 6,
         5, 0, 2,
         4, 7, 8)


def show(board):
    """Print a 9-number board as a 3x3 grid, with 0 shown as _."""
    for row in range(3):
        three = board[row * 3: row * 3 + 3]           # slice out one row
        print(" ".join(str(n) if n != 0 else "_" for n in three))


show(board)

print()
# The pieces, one at a time:
print('Slice of row 1        :', board[3:6])
print('join needs STRINGS    :', " ".join(["5", "0", "2"]))
print('str(5) converts       :', str(5), type(str(5)))
print('"x if test else y"    :', "blank" if 0 == 0 else "tile")

## 16. Reading an error message

You will see errors. Read the **last line first** — it names the problem.

In [ ]:
# Each example is caught so the notebook keeps running.

try:
    campus = {"Main Gate": ["Hostel"]}
    print(campus["Gym"])
except KeyError as e:
    print("KeyError         ->", e, "  (that key is not in the dictionary)")

try:
    path = ["A", "B"]
    print(path[5])
except IndexError as e:
    print("IndexError       ->", e, "  (that position does not exist)")

try:
    print("moves: " + 3)
except TypeError as e:
    print("TypeError        ->", e, "  (wrong type - use str(3) or an f-string)")

try:
    print(undefined_name)
except NameError as e:
    print("NameError        ->", e, "  (typo, or the cell defining it was never run)")

print()
print("Tip: a NameError for something you KNOW you wrote usually means you")
print("     skipped a cell. Run the notebook again from the top.")

## 17. Self-check — predict the output first

For each block below, **write down what you think it prints**, then run the cell.
If you get all six right, you are ready.

**Q1.** What is printed?
```python
path = ["A", "B", "C"]
print(path[-1], len(path) - 1)
```

In [ ]:
path = ["A", "B", "C"]
print(path[-1], len(path) - 1)
# Answer: C 2   -> [-1] is the last item; a 3-place path is 2 moves.

**Q2.** What is printed, and why does `old` not change?
```python
old = ["A"]
new = old + ["B"]
print(old, new)
```

In [ ]:
old = ["A"]
new = old + ["B"]
print(old, new)
# Answer: ['A'] ['A', 'B']   -> "+" builds a NEW list; old is untouched.

**Q3.** How many lines are printed?
```python
seen = set()
for x in ["A", "B", "A", "B"]:
    if x not in seen:
        seen.add(x)
        print(x)
```

In [ ]:
seen = set()
for x in ["A", "B", "A", "B"]:
    if x not in seen:
        seen.add(x)
        print(x)
# Answer: 2 lines, A and B -> the repeats are blocked by the "visited" test.

**Q4.** Which name comes out first?
```python
import heapq
pq = []
heapq.heappush(pq, (5, "X"))
heapq.heappush(pq, (1, "Y"))
print(heapq.heappop(pq))
```

In [ ]:
import heapq
pq = []
heapq.heappush(pq, (5, "X"))
heapq.heappush(pq, (1, "Y"))
print(heapq.heappop(pq))
# Answer: (1, 'Y')   -> the SMALLEST first value leaves first, not the first one added.

**Q5.** What row and column is position 7?
```python
print(7 // 3, 7 % 3)
```

In [ ]:
print(7 // 3, 7 % 3)
# Answer: 2 1  -> row 2, column 1 (the middle of the bottom row).

**Q6.** Why does this stop, and what is printed last?
```python
frontier = [1, 2, 3]
while frontier:
    print(frontier.pop(0), end=" ")
```

In [ ]:
frontier = [1, 2, 3]
while frontier:
    print(frontier.pop(0), end=" ")
print()
# Answer: 1 2 3 -> an EMPTY list counts as False, so the while loop ends by itself.

## 18. Mini exercise — put it all together

This is a miniature of Practical 1: walk a graph, remember what you have seen, and
report the order. It uses **only** the features above — nothing new.

**Do it in two passes.**

1. Run the cell and read it line by line. Every numbered step maps to a section of this notebook.
2. Then scroll down to the empty cell, **cover this one**, and write it again from memory.
   If you can do that, you can write BFS.

In [ ]:
from collections import deque

campus = {
    "Main Gate": ["Hostel", "Admin"],
    "Hostel":    ["Main Gate", "Library"],
    "Admin":     ["Main Gate"],
    "Library":   ["Hostel"]
}


def explore(graph, start):
    """Visit every place reachable from start, nearest first."""
    queue = deque([start])
    visited = set()
    order = []

    while queue:                          # section 8: empty deque = False = stop
        current = queue.popleft()         # section 10: FIFO, so nearest first

        if current in visited:            # section 5: have we been here before?
            continue                      # section 7: skip the rest of this round

        visited.add(current)              # section 5: remember it
        order.append(current)             # section 2: record the order

        for neighbour in graph[current]:  # section 4: look the neighbours up
            if neighbour not in visited:
                queue.append(neighbour)

    return order                          # section 9: hand the answer back


result = explore(campus, "Main Gate")
print("Visit order:", result)
print("Places found:", len(result))

In [ ]:
# Pass 2 - cover the cell above and write explore() again from memory.
# Then run it and check you get the same two lines of output.

**Expected output**

```text
Visit order: ['Main Gate', 'Hostel', 'Admin', 'Library']
Places found: 4
```

If yours matches, you have just written the core of Breadth First Search.

**Extra practice**
1. Change `deque` + `popleft()` to a plain list + `pop()`. The order changes — you have written DFS.
2. Make `explore` return `order` **and** `visited`, then unpack both at the call.
3. Add a `goal` parameter and stop early when `current == goal`.

## 19. Ready-to-start checklist

Tick each one only if you could write it **without looking it up**.

**Basics**
- [ ] Print a value inside a sentence with an f-string
- [ ] Explain what `None` means and test for it with `is None`

**Lists and tuples**
- [ ] Get the last item of a list with `[-1]`
- [ ] Say why `path + [x]` is used instead of `path.append(x)` in a search
- [ ] Say why an 8-puzzle board is a **tuple** and not a list
- [ ] Convert tuple → list → tuple to make a changed copy

**Dictionaries and sets**
- [ ] Store a graph as a dictionary of lists and look up a node's neighbours
- [ ] Loop over a dictionary with `.items()`
- [ ] Use a `set` called `visited` and test membership with `in` / `not in`

**Control flow**
- [ ] Write a goal test with `==`
- [ ] Use `continue` to skip a loop round
- [ ] Explain why `while queue:` stops on its own

**Functions**
- [ ] Write a function that takes parameters and returns **two** values
- [ ] Unpack those two values at the call site

**The data structures the algorithms are built on**
- [ ] `deque` + `popleft()` = queue = **BFS**
- [ ] list + `pop()` = stack = **DFS**
- [ ] `heapq` with `(score, item)` tuples = priority queue = **A\***
- [ ] `min(items, key=lambda x: ...)` to pick the best-scoring item

**Grid maths (Practical 2)**
- [ ] Turn position `p` into a row and column with `p // 3` and `p % 3`
- [ ] Turn a row and column back into a position with `row * 3 + col`
- [ ] Check a move stays on the board with `0 <= r <= 2 and 0 <= c <= 2`
- [ ] Swap two items in a list in one line

---

### If something on this list is unclear

Re-run that section and change the numbers until the output stops surprising you.
Come to the lab with the specific line you did not follow — that is a far better
question than "I don't understand Python".

> **Next:** Practical 1 — *Intelligent Route Planner Using Classical Search Algorithms* (BFS and DFS).